# Experiment 3: Feature Selection and Multicollinearity

## Research Question

Can removing highly redundant predictors reduce model complexity while
maintaining or improving the generalization performance of Linear
Regression on the California Housing dataset?

## Hypothesis

Highly correlated predictors contain overlapping information. Therefore,
removing some redundant predictors may reduce model complexity while
preserving or improving out-of-sample predictive performance.

## Experimental Principle

The feature set will be changed while keeping the following constant:

- Training data
- Validation data
- Test data
- Preprocessing strategy
- Linear Regression model
- Evaluation metrics

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [26]:
df = pd.read_csv("E:\House Price Prediction ML Project\Data\Raw\housing.csv")
df.head()

<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:1: SyntaxWarning: invalid escape sequence '\H'
C:\Users\HP\AppData\Local\Temp\ipykernel_6060\4013469833.py:1: SyntaxWarning: invalid escape sequence '\H'
  df = pd.read_csv("E:\House Price Prediction ML Project\Data\Raw\housing.csv")


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [27]:
X = df.drop(columns=["median_house_value"])
y =df['median_house_value']

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor


#Split the raw data inot train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=12)

#Seprate Numerical Features and Categorical Feature
numerical_features = X_train.select_dtypes(include=('int64', 'float64')).columns
categorical_features = X_train.select_dtypes(include=('str','object')).columns

In [29]:
X_train_pool, X_val, y_train_pool, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

#x_train_pool, y_train_pool -> 80% of original training data, used to train machine learning model.
#x_val, y_val -> 20% of original training data (test_size=0.20), held back to tune hyperparameters and evaluate performance before final testing.


In [30]:
X_train_pool.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
3767,-118.43,34.17,35.0,2922.0,507.0,1130.0,485.0,5.4510,<1H OCEAN
13183,-117.70,33.92,4.0,8301.0,1333.0,3941.0,1236.0,6.2141,<1H OCEAN
9405,-122.53,37.87,20.0,1814.0,282.0,658.0,253.0,7.9977,NEAR BAY
6451,-118.04,34.12,30.0,2170.0,318.0,984.0,309.0,5.6916,INLAND
16822,-122.53,37.65,20.0,4582.0,1124.0,2325.0,1040.0,4.0556,NEAR OCEAN


In [31]:
X_vif = X_train_pool[numerical_features].copy() #creating a copy of the numerical features from the training data to perform VIF analysis and handle missing values.

from sklearn.impute import SimpleImputer #importing SimpleImputer class from sklearn.impute module to handle missing values in the numerical features of the training data.
imputer = SimpleImputer(strategy='median')

X_vif_impute = imputer.fit_transform(X_vif) #Applying median imputation to the numerical features of the training data to handle missing values.

In [32]:
X_vif_imputed = pd.DataFrame(X_vif_impute, columns=numerical_features, index=X_train_pool.index) #creating a DataFrame with the imputed numerical features and the same index as the training data


In [52]:
vif_results = pd.DataFrame() #creating an empty DataFrame to store the results of the Variance Inflation Factor (VIF) analysis for the numerical features in the training data.

vif_results['Features'] = X_vif_imputed.columns

#calculating the Variance Inflation Factor (VIF) for each numerical feature in the training data using the variance_inflation_factor function from the statsmodels library and storing the results in the 'VIF' column of the vif_results DataFrame.
vif_results['VIF_before'] = [variance_inflation_factor(X_vif_imputed.values, i) for i in range(X_vif_imputed.shape[1])] 

vif_results.sort_values(by='VIF_before', ascending=False, inplace=True)

In [53]:
vif_results

,Features,VIF_before
6,households,28.105026
4,total_bedrooms,26.170639
3,total_rooms,12.547515
1,latitude,8.872898
0,longitude,8.729770
5,population,6.721219
7,median_income,1.697774
2,housing_median_age,1.252985


In [36]:
#create feature set
features = [
    'longitude',
    'latitude',
    'housing_median_age',
    'total_rooms',
    'total_bedrooms',
    'population',
    'households',
    'median_income',
    'ocean_proximity'
]

In [50]:
#Remove highly redundant features
feature_set_a = [ feature for feature in X_train_pool.columns if feature != 'households'] 
feature_set_b = [feature for feature in X_train_pool.columns if feature not in ['total_bedrooms']] 
feature_set_c = [feature for feature in X_train_pool.columns if feature not in ['total_rooms']]
feature_set_d = [feature for feature in X_train_pool.columns if feature not in ['population']] 
feature_set_e = [feature for feature in X_train_pool.columns if feature not in ['households', 'total_bedrooms']] 
feature_set_f = [feature for feature in X_train_pool.columns if feature not in ['total_rooms', 'households']] 
feature_set_g = [feature for feature in X_train_pool.columns if feature not in ['households', 'population']] 
feature_set_h = [feature for feature in X_train_pool.columns if feature not in ['total_bedrooms', 'total_rooms']] 
feature_set_i = [feature for feature in X_train_pool.columns if feature not in ['total_bedrooms', 'population']] 
feature_set_j = [feature for feature in X_train_pool.columns if feature not in ['total_rooms', 'population']] 
feature_set_k = [feature for feature in X_train_pool.columns if feature not in ['total_bedrooms', 'households', 'population']] 

In [51]:
feature_sets = {
    "Full": X_train_pool.columns.tolist(),
    "Remove households": feature_set_a,
    "Remove total_bedrooms": feature_set_b,
    "Remove total_rooms": feature_set_c,
    "Remove population": feature_set_d,
    "Remove households and total_bedrooms": feature_set_e,
    "Remove total_rooms and households": feature_set_f,
    "Remove households and population": feature_set_g,
    "Remove total_bedrooms and total_rooms": feature_set_h,
    "Remove total_bedrooms and population": feature_set_i,
    "Remove total_rooms and population": feature_set_j,
    "Remove total_bedrooms, households, and population": feature_set_k
}

feature_sets

{'Full': ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'total_bedrooms',
  'population',
  'households',
  'median_income',
  'ocean_proximity'],
 'Remove households': ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'total_bedrooms',
  'population',
  'median_income',
  'ocean_proximity'],
 'Remove total_bedrooms': ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'population',
  'households',
  'median_income',
  'ocean_proximity'],
 'Remove total_rooms': ['longitude',
  'latitude',
  'housing_median_age',
  'total_bedrooms',
  'population',
  'households',
  'median_income',
  'ocean_proximity'],
 'Remove population': ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'total_bedrooms',
  'households',
  'median_income',
  'ocean_proximity'],
 'Remove households and total_bedrooms': ['longitude',
  'latitude',
  'housing_median_age',
  'total_rooms',
  'population',
  'median_income',
  'ocean_pro

In [59]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np


def calculate_vif(X):

    numerical_features = X.select_dtypes(
        include=["int64", "float64"]
    ).columns

    X_numeric = X[numerical_features].copy()

    imputer = SimpleImputer(strategy="median")

    X_imputed = imputer.fit_transform(X_numeric)

    X_imputed = pd.DataFrame(
        X_imputed,
        columns=numerical_features
    )

    vif = pd.DataFrame()

    vif["Feature"] = X_imputed.columns

    vif["VIF"] = [
        variance_inflation_factor(
            X_imputed.values,
            i
        )
        for i in range(X_imputed.shape[1])
    ]

    return vif.sort_values(
        by="VIF",
        ascending=False
    )

In [60]:
vif_results = []

for name, features in feature_sets.items():

    X_subset = X_train_pool[features]

    vif = calculate_vif(X_subset)

    max_vif = vif["VIF"].max()

    vif_results.append({
        "Feature_Set": name,
        "Number_of_Features": len(features),
        "Max_VIF": max_vif
    })

vif_results_df = pd.DataFrame(vif_results)

vif_results_df

,Feature_Set,Number_of_Features,Max_VIF
0,Full,9,28.105026
1,Remove households,8,12.499407
2,Remove total_bedrooms,8,12.078914
3,Remove total_rooms,8,27.997269
4,Remove population,8,24.862424
5,Remove households and total_bedrooms,7,8.378570
6,Remove total_rooms and households,7,8.122502
7,Remove households and population,7,11.037251
8,Remove total_bedrooms and total_rooms,7,8.141261
9,Remove total_bedrooms and population,7,9.545829


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

model_results = []

for name, features in feature_sets.items():

    X_train_subset = X_train_pool[features]
    X_val_subset = X_val[features]

    # Create preprocessing + model for this feature subset
    model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
            ])

    # Create preprocessing + model for this feature subset
    numerical_cols = X_train_subset.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = X_train_subset.select_dtypes(include=['object']).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), numerical_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ])
    
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ])
    
    model.fit(X_train_subset, y_train_pool)

    y_train_pred = model.predict(X_train_subset)
    y_val_pred = model.predict(X_val_subset)

    train_rmse = root_mean_squared_error(y_train_pool, y_train_pred)
    val_rmse = root_mean_squared_error(y_val, y_val_pred)

    train_mae = mean_absolute_error(y_train_pool, y_train_pred)
    val_mae = mean_absolute_error(y_val, y_val_pred)

    train_r2 = r2_score(y_train_pool, y_train_pred)
    val_r2 = r2_score(y_val, y_val_pred)

    model_results.append({
        "Feature_Set": name,
        "Number_of_Features": len(features),
        "Train_RMSE": train_rmse,
        "Val_RMSE": val_rmse,
        "Train_MAE": train_mae,
        "Val_MAE": val_mae,
        "Train_R2": train_r2,
        "Val_R2": val_r2
    })

model_results_df = pd.DataFrame(model_results)

model_results_df

TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'LinearRegression()' (type <class 'sklearn.linear_model._base.LinearRegression'>) doesn't

In [ ]:
experiment_3_results = model_results_df.merge(
    vif_results_df[
        [
            "Feature_Set",
            "Max_VIF"
        ]
    ],
    on="Feature_Set"
)

experiment_3_results

In [ ]:
experiment_3_results = experiment_3_results[
    [
        "Feature_Set",
        "Number_of_Features",
        "Max_VIF",
        "Train_RMSE",
        "Validation_RMSE",
        "Train_MAE",
        "Validation_MAE",
        "Train_R2",
        "Validation_R2"
    ]
]

experiment_3_results

In [ ]:
experiment_3_results.sort_values(
    by="Validation_RMSE"
)

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    experiment_3_results["Feature_Set"],
    experiment_3_results["Max_VIF"]
)

plt.xlabel("Feature Set")
plt.ylabel("Maximum VIF")
plt.title("Effect of Feature Selection on Multicollinearity")

plt.xticks(
    rotation=75,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    experiment_3_results["Feature_Set"],
    experiment_3_results["Validation_RMSE"]
)

plt.xlabel("Feature Set")
plt.ylabel("Validation RMSE")
plt.title("Effect of Feature Selection on Validation RMSE")

plt.xticks(
    rotation=75,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    experiment_3_results["Max_VIF"],
    experiment_3_results["Validation_RMSE"]
)

plt.xlabel("Maximum VIF")
plt.ylabel("Validation RMSE")
plt.title("Multicollinearity vs Predictive Performance")

plt.show()